In [ ]:
import numpy as np
import pandas as pd

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score


In [105]:
df = pd.read_parquet('../data/processed/abcd_first_msgs.parquet')
# df_sample = pd.read_json('../raw_data/abcd_sample.json')


In [106]:
df

,split,id,flow,subflow,first_msgs,refund,refund_label,n_words
0,train,3592,product_defect,return_size,"Hi! I need to return an item, can you help me with that?",0,1,13
1,train,9489,product_defect,refund_status,just wanted to check on the status of a refund,0,0,10
2,train,3695,storewide_query,timing_4,HEY HO! I've got a promo code and I want to know when they expire. I'd like to use it to buy some hats for my cat.,0,0,27
3,train,5798,shipping_issue,manage,"Hello, I would like to change my shipping deatails as they have changed recently due to a move",0,1,18
4,train,3647,storewide_query,pricing_3,Hello. I have a really cool party coming up. And I need some new clothes ASAP. I am thinking of ordering them to come by overnight shipping Do you know how much that costs?,0,0,34
...,...,...,...,...,...,...,...,...
10037,test,1622,order_issue,manage_upgrade,HI I want to upgrade an order I have with you to overnight shipping,0,0,14
10038,test,3293,manage_account,status_credit_missing,"Hi, I think I am missing a $40 credit from a subscription refund.",0,0,13
10039,test,8492,subscription_inquiry,manage_extension,I would like an extension on my subscription.,0,0,8
10040,test,189,storewide_query,pricing_1,Hi! How much do you charge to gift wrap items?,0,0,10


In [108]:
# REFUND_SUBFLOWS = {
#     "refund_initiate",
#     "return_size",
#     "return_color",
#     "return_stain",
#     "manage_dispute_bill",
#     "bad_price_competitor",
#     "status_mystery_fee",
#     "refund_update",
#     "manage_cancel",
# }

# df["refund_request"] = df["subflow"].isin(REFUND_SUBFLOWS).astype(int)
# print(df["refund_request"].value_counts())

In [109]:
TARGET = "refund_label"
train = df[df["split"] == "train"]
val = df[df["split"] == "dev"]

X_train = train["first_msgs"]
y_train = train[TARGET]

X_val = val["first_msgs"]
y_val = val[TARGET]

In [110]:
# Share of refund vs not refund in the training data
print(y_train.value_counts(normalize=True))

# A model that ignores the text and always predicts the most common answer
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
y_pred = dummy.predict(X_val)

# Score it on the validation set (precision and recall are for the refund class, 1)
print("accuracy:", accuracy_score(y_val, y_pred))
print("precision:", precision_score(y_val, y_pred, zero_division=0))
print("recall:", recall_score(y_val, y_pred))

refund_label
0    0.776948
1    0.223052
Name: proportion, dtype: float64
accuracy: 0.7719123505976095
precision: 0.0
recall: 0.0


In [111]:
models = {
    "count + none": make_pipeline(
        CountVectorizer(),
        LogisticRegression(max_iter=1000)
    ),
    "count + balanced": make_pipeline(
        CountVectorizer(),
        LogisticRegression(max_iter=1000, class_weight="balanced")
    ),
    "tfidf + none": make_pipeline(
        TfidfVectorizer(),
        LogisticRegression(max_iter=1000)
    ),
    "tfidf + balanced": make_pipeline(
        TfidfVectorizer(),
        LogisticRegression(max_iter=1000, class_weight="balanced")
    )
}

In [112]:
# for name, pipe in models.items():
#     pipe.fit(X_train, y_train)
#     print(f"\n=== {name} ===")
#     print(classification_report(y_val, pipe.predict(X_val)))

In [113]:
results = []

for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_val)

    results.append({
        "model": name,
        "precision": precision_score(y_val, y_pred),
        "recall": recall_score(y_val, y_pred),
        "f1": f1_score(y_val, y_pred)
    })
    results_df = pd.DataFrame(results).sort_values("f1", ascending=False)


In [114]:
results_df


,model,precision,recall,f1
0,count + none,0.935484,0.886463,0.910314
3,tfidf + balanced,0.837209,0.943231,0.887064
1,count + balanced,0.835938,0.934498,0.882474
2,tfidf + none,0.935000,0.816594,0.871795


In [122]:
best_name = results_df.iloc[0]["model"]
best_model = models[best_name]

In [121]:
df[df["refund_label"] == 1]["subflow"].unique().tolist()

['return_size',
 'manage',
 'status_quantity',
 'refund_update',
 'refund_initiate',
 'return_color',
 'bad_price_competitor',
 'status_mystery_fee',
 'cost',
 'manage_dispute_bill',
 'status_service_added',
 'manage_cancel',
 'return_stain']